# Detección de QRs — versión dinámica

Esta versión elimina la dependencia de bandas blancas fijas y de medidas reales predefinidas de la mesa.

La idea nueva es:
1. Buscar QRs en toda la imagen, no solo en zonas horizontales.
2. Ordenar los QRs detectados por posición relativa: TL, TR, BR, BL.
3. Construir una homografía dinámica usando las propias posiciones detectadas.
4. Rectificar la zona de trabajo a una vista cenital normalizada.

Así la imagen puede estar tomada desde distinto ángulo, distancia o rotación.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os
import glob

IMAGE_PATH  = "../data/multimedia_01.jpg"
DATASET_PATH = "../data"
N_QRS = 2

## Módulo 1: utilidades geométricas dinámicas

In [ ]:
def order_points_clockwise(points):
    """
    Ordena puntos como TL, TR, BR, BL usando solo su posición en la imagen.
    Funciona aunque la mesa esté girada o desplazada dentro de la foto.
    """
    pts = np.array(points, dtype=np.float32)

    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1).reshape(-1)

    ordered = np.zeros((4, 2), dtype=np.float32)
    ordered[0] = pts[np.argmin(s)]       # TL: x+y menor
    ordered[2] = pts[np.argmax(s)]       # BR: x+y mayor
    ordered[1] = pts[np.argmin(diff)]    # TR: y-x menor
    ordered[3] = pts[np.argmax(diff)]    # BL: y-x mayor
    return ordered


def label_qrs_by_position(candidates):
    """
    Recibe candidatos QR y devuelve un diccionario con etiquetas TL/TR/BR/BL.
    No asume coordenadas reales ni tamaño fijo de mesa.
    """
    if len(candidates) == 0:
        return {}

    centers = np.array([c['center'] for c in candidates], dtype=np.float32)

    if len(candidates) >= 4:
        # Nos quedamos con los 4 candidatos más externos respecto al centro global.
        center_global = centers.mean(axis=0)
        dist = np.linalg.norm(centers - center_global, axis=1)
        idx = np.argsort(dist)[-4:]
        selected = [candidates[i] for i in idx]
        ordered = order_points_clockwise([c['center'] for c in selected])
        labels = ['TL', 'TR', 'BR', 'BL']

        qrs = {}
        for label, pt in zip(labels, ordered):
            j = int(np.argmin([np.linalg.norm(np.array(c['center']) - pt) for c in selected]))
            c = selected[j]
            qrs[label] = c
        return qrs

    # Con 3 QRs no podemos tener homografía completa, pero sí etiquetar aproximadamente.
    qrs = {}
    for c in candidates:
        x, y = c['center']
        score_tl = x + y
        score_br = x + y
        score_tr = y - x
        score_bl = x - y
        c['_scores'] = {'TL': score_tl, 'BR': score_br, 'TR': score_tr, 'BL': score_bl}

    remaining = candidates.copy()
    tl = min(remaining, key=lambda c: c['_scores']['TL']); qrs['TL'] = tl; remaining.remove(tl)
    if remaining:
        br = max(remaining, key=lambda c: c['_scores']['BR']); qrs['BR'] = br; remaining.remove(br)
    if remaining:
        c = remaining[0]
        qrs['TR' if (c['center'][0] > qrs['TL']['center'][0]) else 'BL'] = c

    for c in candidates:
        c.pop('_scores', None)
    return qrs


print("Módulo 1 listo.")

## Módulo 2: detección de la mesa y máscara

Detecta el polígono del papel blanco y rellena el exterior con un valor uniforme (127)
antes de buscar candidatos visuales. Así `findContours` no genera candidatos fuera de la mesa.

In [ ]:
def detect_table_polygon(img_bgr, min_white_area_ratio=0.08):
    """
    Estima el polígono exterior del papel blanco usando umbralización HSV.
    Devuelve 4 puntos ordenados TL/TR/BR/BL, o None si no detecta la mesa.
    """
    hsv  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    H, W = img_bgr.shape[:2]

    # Papel blanco: brillo alto y saturación baja
    mask = cv2.inRange(hsv, np.array([0, 0, 120]), np.array([179, 80, 255]))

    k = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k, iterations=1)

    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    clean   = np.zeros_like(mask)
    min_area = H * W * min_white_area_ratio / 20
    for cnt in cnts:
        if cv2.contourArea(cnt) >= min_area:
            cv2.drawContours(clean, [cnt], -1, 255, -1)

    ys, xs = np.where(clean > 0)
    if len(xs) < H * W * min_white_area_ratio:
        return None

    pts  = np.column_stack([xs, ys]).astype(np.float32)
    rect = cv2.minAreaRect(pts)
    box  = cv2.boxPoints(rect).astype(np.float32)
    return order_points_clockwise(box)


def apply_table_mask(img_bgr, table_polygon, fill_value=127):
    """
    Rellena con fill_value todo lo que esté fuera del polígono de la mesa.
    Con fill_value=127 (gris medio) no se generan gradientes en el borde
    y findContours no produce candidatos fuera del papel.
    """
    mask   = np.zeros(img_bgr.shape[:2], dtype=np.uint8)
    cv2.fillPoly(mask, [table_polygon.astype(np.int32)], 255)
    result = img_bgr.copy()
    result[mask == 0] = fill_value
    return result


print("Módulo 2 listo.")

## Módulo 3: detección de QRs en toda la imagen

In [ ]:
def _try_decode_qrs(img_bgr, scales=(1, 2, 3, 4, 6)):
    """
    Intenta detectar QRs reales con OpenCV usando varias escalas.
    Devuelve candidatos con centro, tamaño, datos decodificados y esquinas.
    """
    detector = cv2.QRCodeDetector()
    candidates = []

    for scale in scales:
        if scale == 1:
            work = img_bgr
        else:
            work = cv2.resize(img_bgr, None, fx=scale, fy=scale,
                              interpolation=cv2.INTER_CUBIC)

        variants = [work]
        gray = cv2.cvtColor(work, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(4, 4))
        enh = clahe.apply(gray)
        variants.append(cv2.cvtColor(enh, cv2.COLOR_GRAY2BGR))

        for variant in variants:
            ret, decoded, pts, _ = detector.detectAndDecodeMulti(variant)
            if not ret or pts is None:
                continue

            for i, p in enumerate(pts):
                p = (p / scale).astype(np.float32)
                cx, cy = p[:, 0].mean(), p[:, 1].mean()
                size = float(np.mean([
                    np.linalg.norm(p[0] - p[1]),
                    np.linalg.norm(p[1] - p[2]),
                    np.linalg.norm(p[2] - p[3]),
                    np.linalg.norm(p[3] - p[0]),
                ]))
                data = decoded[i] if decoded and i < len(decoded) else ''
                candidates.append({
                    'center': (float(cx), float(cy)),
                    'size': size,
                    'data': data,
                    'corners': p,
                    'method': 'decode'
                })

    return _merge_close_candidates(candidates)


def _merge_close_candidates(candidates, dist_factor=1.5):
    """Elimina candidatos repetidos detectados a distintas escalas."""
    merged = []
    for c in sorted(candidates, key=lambda z: z.get('size', 0), reverse=True):
        cx, cy = c['center']
        duplicated = False
        for m in merged:
            mx, my = m['center']
            max_dist = dist_factor * max(c.get('size', 20), m.get('size', 20))
            if np.hypot(cx - mx, cy - my) < max_dist:
                duplicated = True
                break
        if not duplicated:
            merged.append(c)
    return merged


def _visual_qr_candidates(img_bgr,
                          min_size_ratio=0.006,
                          max_size_ratio=0.08,
                          min_white_ring=0.50,
                          min_edge_density=0.10):
    """
    Fallback visual cuando el QR no se puede decodificar.

    Busca cuadrados pequeños con mucha textura interna y fondo claro alrededor.
    Se recomienda pasar la imagen ya enmascarada con apply_table_mask() para
    que el exterior de la mesa no genere candidatos.
    """
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    H, W = gray.shape
    diag = np.hypot(H, W)
    min_size = int(diag * min_size_ratio)
    max_size = int(diag * max_size_ratio)

    blur = cv2.GaussianBlur(gray, (3, 3), 0)
    enh = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(4, 4)).apply(blur)

    binaries = [
        cv2.adaptiveThreshold(enh, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                              cv2.THRESH_BINARY_INV, 21, 5),
        cv2.adaptiveThreshold(enh, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                              cv2.THRESH_BINARY_INV, 21, 7),
        cv2.threshold(enh, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    ]

    candidates = []
    for binary in binaries:
        cnts, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        for cnt in cnts:
            x, y, w, h = cv2.boundingRect(cnt)
            size = max(w, h)
            if size < min_size or size > max_size:
                continue

            aspect = w / (h + 1e-6)
            if not (0.75 <= aspect <= 1.33):
                continue

            area = cv2.contourArea(cnt)
            fill = area / (w * h + 1e-6)
            if fill < 0.05:
                continue

            pad = int(size * 1.4)
            x1 = max(0, x - pad); y1 = max(0, y - pad)
            x2 = min(W, x + w + pad); y2 = min(H, y + h + pad)

            ring = gray[y1:y2, x1:x2].copy()
            ring[max(0, y-y1):min(y+h, y2)-y1,
                 max(0, x-x1):min(x+w, x2)-x1] = 0
            vals = ring[ring > 0]
            white_ring = np.mean(vals > 135) if vals.size else 0
            if white_ring < min_white_ring:
                continue

            roi = gray[y:y+h, x:x+w]
            edges = cv2.Canny(roi, 40, 120)
            edge_density = np.mean(edges > 0)
            if edge_density < min_edge_density:
                continue

            score = white_ring + edge_density - abs(aspect - 1.0) * 0.5
            corners = np.array([[x, y], [x+w, y], [x+w, y+h], [x, y+h]], dtype=np.float32)
            candidates.append({
                'center': (float(x + w/2), float(y + h/2)),
                'size': float(size),
                'data': None,
                'corners': corners,
                'method': 'visual',
                'score': float(score)
            })

    candidates = sorted(candidates, key=lambda c: c.get('score', 0), reverse=True)
    return _merge_close_candidates(candidates)


def detect_qrs(image_path, show=True):
    """
    Detección dinámica de QRs.
    Aplica máscara de mesa antes de la búsqueda visual para eliminar
    candidatos de ruido fuera del papel.
    """
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(image_path)

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Enmascarar exterior de la mesa antes de buscar candidatos visuales
    table_polygon = detect_table_polygon(img_bgr)
    if table_polygon is not None:
        img_for_visual = apply_table_mask(img_bgr, table_polygon, fill_value=127)
    else:
        img_for_visual = img_bgr

    decoded = _try_decode_qrs(img_bgr)               # imagen original
    visual  = _visual_qr_candidates(img_for_visual)  # imagen enmascarada

    candidates = _merge_close_candidates(decoded + visual)
    qrs = label_qrs_by_position(candidates)

    if show:
        _visualize_qrs_dynamic(img_rgb, qrs, candidates, os.path.basename(image_path))

    return img_rgb, qrs


def _visualize_qrs_dynamic(img_rgb, qrs, candidates, title):
    overlay = img_rgb.copy()
    colors = {'TL':(0,255,120), 'TR':(0,180,255), 'BR':(255,100,0), 'BL':(255,220,0)}

    for c in candidates:
        x, y = map(int, c['center'])
        s = int(c.get('size', 25))
        cv2.rectangle(overlay, (x-s, y-s), (x+s, y+s), (180, 180, 180), 2)

    for name, info in qrs.items():
        cx, cy = map(int, info['center'])
        s = int(info.get('size', 25)) + 12
        col = colors.get(name, (255, 255, 0))
        cv2.rectangle(overlay, (cx-s, cy-s), (cx+s, cy+s), col, 4)
        cv2.circle(overlay, (cx, cy), 7, col, -1)
        txt = f"{name} ({info.get('method','')})"
        cv2.putText(overlay, txt, (cx-40, cy-s-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.75, col, 2)

    if len(qrs) >= 3:
        pts = np.array([v['center'] for v in qrs.values()], dtype=np.int32)
        hull = cv2.convexHull(pts)
        cv2.polylines(overlay, [hull], True, (255, 255, 0), 3)

    plt.figure(figsize=(10, 10))
    plt.imshow(overlay)
    plt.title(f"{title} | QRs etiquetados: {len(qrs)} | candidatos: {len(candidates)}")
    plt.axis('off')
    plt.show()


print("Módulo 3 listo.")

## Módulo 4: rectificación dinámica de la zona de trabajo

In [ ]:
def compute_dynamic_homography(qrs, output_size=(1000, 1400)):
    """
    Calcula una homografía imagen → vista cenital normalizada.

    No necesita medidas físicas reales. El resultado queda en un sistema de coordenadas
    normalizado de tamaño output_size = (ancho, alto).

    Con 4 QRs: homografía completa.
    Con 3 QRs: transformación afín aproximada.
    """
    out_w, out_h = output_size

    required4 = ['TL', 'TR', 'BR', 'BL']
    if all(k in qrs for k in required4):
        src = np.array([qrs[k]['center'] for k in required4], dtype=np.float32)
        dst = np.array([[0, 0], [out_w, 0], [out_w, out_h], [0, out_h]], dtype=np.float32)
        H, _ = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)
        return H, required4

    # Fallback con 3 puntos: no corrige toda la perspectiva, pero estabiliza escala/rotación.
    available = [k for k in ['TL', 'TR', 'BR', 'BL'] if k in qrs]
    if len(available) < 3:
        raise ValueError(f"Solo {len(available)} QRs etiquetados. Se necesitan al menos 3.")

    dst_map = {
        'TL': (0, 0),
        'TR': (out_w, 0),
        'BR': (out_w, out_h),
        'BL': (0, out_h),
    }
    src = np.array([qrs[k]['center'] for k in available[:3]], dtype=np.float32)
    dst = np.array([dst_map[k] for k in available[:3]], dtype=np.float32)

    A = cv2.getAffineTransform(src, dst)
    H = np.vstack([A, [0, 0, 1]]).astype(np.float32)
    return H, available[:3]


def warp_workspace(image_path, qrs, output_size=(1000, 1400), show=True):
    """
    Rectifica la mesa/zona de trabajo a una vista normalizada.
    Esta imagen rectificada es la que conviene usar después para detectar la separación entre gomas.
    """
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(image_path)

    H, used = compute_dynamic_homography(qrs, output_size=output_size)
    warped = cv2.warpPerspective(img_bgr, H, output_size)
    warped_rgb = cv2.cvtColor(warped, cv2.COLOR_BGR2RGB)

    if show:
        plt.figure(figsize=(8, 10))
        plt.imshow(warped_rgb)
        plt.title(f"Vista normalizada usando QRs: {used}")
        plt.axis('off')
        plt.show()

    return warped_rgb, H, used


def transform_point(pt_px, H):
    p = np.array([pt_px[0], pt_px[1], 1.0], dtype=np.float32)
    q = H @ p
    return float(q[0] / q[2]), float(q[1] / q[2])


def measure_gap_normalized(pt1_px, pt2_px, H):
    """
    Mide una separación en unidades normalizadas de la imagen rectificada.
    Para convertir a mm hace falta una escala real conocida.
    """
    p1 = transform_point(pt1_px, H)
    p2 = transform_point(pt2_px, H)
    d = np.hypot(p2[0] - p1[0], p2[1] - p1[1])
    return d, p1, p2


print("Módulo 4 listo.")

## Test sobre una imagen

In [ ]:
img_rgb, qrs = detect_qrs(IMAGE_PATH, show=True)

print("Resultado:")
for k, v in sorted(qrs.items()):
    print(f"  {k}: center={v['center']}  size={v['size']:.1f}px  method={v.get('method')}  data='{v.get('data')}'")

if len(qrs) >= 3:
    warped_rgb, H_norm, used = warp_workspace(IMAGE_PATH, qrs, output_size=(1000, 1400), show=True)
    print(f"Homografía dinámica calculada con: {used}")
else:
    print("No hay suficientes QRs para rectificar la zona de trabajo.")

## Módulo 5: medición

La medición ya no depende de `REAL_COORDS_MM`.

Primero se rectifica la imagen a una vista normalizada. Después, cuando se detecten los bordes de las gomas, la separación se puede medir en esa imagen rectificada.

Para convertir esa distancia a milímetros hay dos opciones:
- usar el tamaño físico real del QR impreso;
- o usar una distancia real conocida entre dos QRs.

Sin una escala física real, el sistema puede medir de forma consistente en unidades normalizadas, pero no en mm absolutos.

In [ ]:
def normalized_to_mm(distance_norm, scale_mm_per_norm_px):
    """
    Convierte una distancia de la imagen rectificada a mm.
    scale_mm_per_norm_px debe calibrarse con una referencia física real.
    """
    return distance_norm * scale_mm_per_norm_px


# Ejemplo de uso futuro:
# gap_norm, p1_norm, p2_norm = measure_gap_normalized(borde_izq_px, borde_der_px, H_norm)
# gap_mm = normalized_to_mm(gap_norm, scale_mm_per_norm_px)
# print(f"Separación: {gap_mm:.2f} mm")

print("Módulo 5 listo.")

## Evaluación sobre todo el dataset

In [ ]:
image_paths = sorted(glob.glob(os.path.join(DATASET_PATH, "*.jpg")))
print(f"Imágenes: {len(image_paths)}")

summary = []
for path in image_paths:
    fname = os.path.basename(path)
    try:
        _, qrs_i = detect_qrs(path, show=False)
        n = len(qrs_i)
        decoded = sum(1 for v in qrs_i.values() if v.get('data'))
        ok = n >= N_QRS
        summary.append((fname, n, decoded, ok))
        tag = 'OK  ' if ok else 'FAIL'
        print(f"  [{tag}] {fname}: {n} QRs ({decoded} decoded) {[k for k in qrs_i]}")
    except Exception as e:
        summary.append((fname, 0, 0, False))
        print(f"  [ERR ] {fname}: {e}")

n_ok = sum(1 for *_, ok in summary if ok)
print(f"Detección exitosa: {n_ok}/{len(image_paths)} ({100*n_ok/max(len(image_paths),1):.1f}%)")